# Zillow Prize 1 Human Baseline Work

Manual notebook artifact for building `tc1_human.json`.
This records the current notebook-aligned Zillow TC1 human baseline.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path.home() / "Downloads" / "zillow-prize-1"
assert DATA_DIR.exists(), f"Missing data directory: {DATA_DIR}"

train_2016 = pd.read_csv(DATA_DIR / "train_2016_v2.csv")
train_2017 = pd.read_csv(DATA_DIR / "train_2017.csv")
properties_2016 = pd.read_csv(DATA_DIR / "properties_2016.csv", low_memory=False)
properties_2017 = pd.read_csv(DATA_DIR / "properties_2017.csv", low_memory=False)
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
data_dictionary_path = DATA_DIR / "zillow_data_dictionary.xlsx"

{
    "data_dir": str(DATA_DIR),
    "train_2016": train_2016.shape,
    "train_2017": train_2017.shape,
    "properties_2016": properties_2016.shape,
    "properties_2017": properties_2017.shape,
    "sample_submission": sample_submission.shape,
    "data_dictionary_exists": data_dictionary_path.exists(),
}


## Median Impute Numeric Property Columns

Before encoding categoricals, fill missing values for the numeric property columns where a median-based fill is plausible.

Included here:
- size/area fields that are not obvious absence indicators
- room and bathroom counts
- year and tax value fields

Excluded for now:
- absence-style numeric columns such as fireplaces, garages, pools, sheds, and basements; these will get constant fills below
- categorical code columns such as `*typeid`
- region identifiers such as `regionid*`, `fips`, and census/block identifiers
- text/flag columns such as `propertycountylandusecode`, `propertyzoningdesc`, `hashottuborspa`, `fireplaceflag`, and `taxdelinquencyflag`

In [ ]:
median_impute_columns = [
    "bathroomcnt",
    "bedroomcnt",
    "calculatedbathnbr",
    "threequarterbathnbr",
    "finishedfloor1squarefeet",
    "calculatedfinishedsquarefeet",
    "finishedsquarefeet6",
    "finishedsquarefeet12",
    "finishedsquarefeet13",
    "finishedsquarefeet15",
    "finishedsquarefeet50",
    "fullbathcnt",
    "lotsizesquarefeet",
    "numberofstories",
    "roomcnt",
    "unitcnt",
    "yearbuilt",
    "structuretaxvaluedollarcnt",
    "taxvaluedollarcnt",
    "assessmentyear",
    "landtaxvaluedollarcnt",
    "taxamount",
    "taxdelinquencyyear",
]

properties_2016[median_impute_columns] = properties_2016[median_impute_columns].fillna(
    properties_2016[median_impute_columns].median()
)

properties_2016[median_impute_columns].isna().sum().sort_values(ascending=False).head()


## Constant Impute Absence-Style Columns

Some columns look much more like `missing = not present` than `missing = unknown numeric value`.

Use grouped constant fills where the same imputation is natural:
- fill `0` for absence-style numeric columns
- fill `FALSE` for boolean-like string flags that are only observed as `true`
- fill `N` for `taxdelinquencyflag`, where non-missing values are observed as `Y`

In [ ]:
zero_fill_columns = [
    "basementsqft",
    "fireplacecnt",
    "garagecarcnt",
    "garagetotalsqft",
    "poolcnt",
    "poolsizesum",
    "pooltypeid10",
    "pooltypeid2",
    "pooltypeid7",
    "yardbuildingsqft17",
    "yardbuildingsqft26",
]
false_fill_columns = [
    "hashottuborspa",
    "fireplaceflag",
]
no_fill_columns = [
    "taxdelinquencyflag",
]

properties_2016[zero_fill_columns] = properties_2016[zero_fill_columns].fillna(0)
properties_2016[false_fill_columns] = properties_2016[false_fill_columns].fillna("FALSE")
properties_2016[no_fill_columns] = properties_2016[no_fill_columns].fillna("N")

{
    "zero_fill_columns": zero_fill_columns,
    "false_fill_columns": false_fill_columns,
    "no_fill_columns": no_fill_columns,
    "remaining_missing_after_constant_fill": properties_2016[
        zero_fill_columns + false_fill_columns + no_fill_columns
    ].isna().sum().sum(),
}


## Impute Remaining Missing Property Columns With `-1`

After the median and absence-style constant fills above, the remaining missing property columns are mostly code-like ids, location identifiers, or high-cardinality/string-like fields.

For the current human baseline, fill the remaining missing property columns with a shared `-1` sentinel before one-hot encoding and joining.

In [ ]:
negative_one_fill_columns = [
    "airconditioningtypeid",
    "architecturalstyletypeid",
    "buildingclasstypeid",
    "buildingqualitytypeid",
    "decktypeid",
    "fips",
    "heatingorsystemtypeid",
    "latitude",
    "longitude",
    "propertycountylandusecode",
    "propertylandusetypeid",
    "propertyzoningdesc",
    "rawcensustractandblock",
    "regionidcity",
    "regionidcounty",
    "regionidneighborhood",
    "regionidzip",
    "storytypeid",
    "typeconstructiontypeid",
    "censustractandblock",
]

properties_2016[negative_one_fill_columns] = properties_2016[negative_one_fill_columns].fillna(-1)

{
    "negative_one_fill_columns": negative_one_fill_columns,
    "remaining_missing_after_negative_one_fill": properties_2016[
        negative_one_fill_columns
    ].isna().sum().sum(),
}


## One-Hot Encode Nominal Categoricals

One-hot encode the low/medium-cardinality categorical codes that look nominal rather than ordinal.

Included here:
- `airconditioningtypeid`
- `architecturalstyletypeid`
- `buildingclasstypeid`
- `heatingorsystemtypeid`
- `propertylandusetypeid`
- `storytypeid`
- `typeconstructiontypeid`

Excluded for now:
- `buildingqualitytypeid` because it looks ordinal
- `regionid*`, `propertycountylandusecode`, and `propertyzoningdesc` because they are high-cardinality

Use `dummy_na=False` so that no separate NaN indicator columns are created; missing values have already been filled with `-1` and will be represented by the corresponding `-1` dummy category.

In [ ]:
nominal_one_hot_columns = [
    "airconditioningtypeid",
    "architecturalstyletypeid",
    "buildingclasstypeid",
    "heatingorsystemtypeid",
    "propertylandusetypeid",
    "storytypeid",
    "typeconstructiontypeid",
]

properties_2016[nominal_one_hot_columns].nunique(dropna=True).sort_values()

properties_2016 = pd.get_dummies(
    properties_2016,
    columns=nominal_one_hot_columns,
    dummy_na=False,
)

{
    "selected_columns": nominal_one_hot_columns,
    "properties_2016_shape_after_one_hot": properties_2016.shape,
}


## Join Train With Property Lookup

Join `train_2016_v2.csv` with `properties_2016.csv` on `parcelid`.

In [ ]:
train_with_properties_2016 = train_2016.merge(
    properties_2016,
    how="left",
    on="parcelid",
)

{
    "train_2016": train_2016.shape,
    "properties_2016": properties_2016.shape,
    "train_with_properties_2016": train_with_properties_2016.shape,
    "joined_columns": len(train_with_properties_2016.columns),
}


## Trim Logerror Outliers

Trim the bottom 0.5% and top 0.5% of `logerror` from the joined 2016 training frame.

In [ ]:
logerror_lower = train_with_properties_2016["logerror"].quantile(0.005)
logerror_upper = train_with_properties_2016["logerror"].quantile(0.995)

train_with_properties_2016_trimmed = train_with_properties_2016[
    train_with_properties_2016["logerror"].between(logerror_lower, logerror_upper)
].copy()

{
    "logerror_lower_0p5pct": float(logerror_lower),
    "logerror_upper_99p5pct": float(logerror_upper),
    "rows_before_trim": len(train_with_properties_2016),
    "rows_after_trim": len(train_with_properties_2016_trimmed),
}
